In [8]:
import pandas as pd
import xarray as xr
import numpy as np

def add_bio10(csv_path, nc_path,
              year_col="year", lat_col="decimalLatitude", lon_col="decimalLongitude",
              out_csv=None, method="nearest", engine=None):
    """
    Add a BIO10 column to a CSV using a gridded NetCDF (0.5°) with annual data.

    method: 'nearest' (gridpoint match) or anything accepted by xarray.interp ('linear', etc.)
    engine: force backend, e.g. 'netcdf4', 'h5netcdf', or 'scipy'
    """
    df = pd.read_csv(csv_path)

    # Open dataset
    ds = xr.open_dataset(nc_path, engine=engine)

    var_name = "BIO15"

    # ---- Time/Year handling ----
    if "year" in ds.coords:
        # Dataset already indexed by integer years
        years_da = xr.DataArray(df[year_col].astype(int).values, dims="rows")
        var_yr = ds[var_name].sel(year=years_da)
    elif "time" in ds.coords or "time" in ds.dims:
        # Collapse to annual means if needed, then index by year
        # (If already annual, .groupby won't change values.)
        ds_year = ds[var_name].groupby("time.year").mean("time")
        years_da = xr.DataArray(df[year_col].astype(int).values, dims="rows")
        var_yr = ds_year.sel(year=years_da)
    else:
        raise KeyError("No 'year' or 'time' coordinate found. Inspect ds.coords / ds.dims.")

    # ---- Spatial dims ----
    lat_name = "lat" if "lat" in ds.dims else ("latitude" if "latitude" in ds.dims else None)
    lon_name = "lon" if "lon" in ds.dims else ("longitude" if "longitude" in ds.dims else None)
    if lat_name is None or lon_name is None:
        raise KeyError(f"Couldn't find lat/lon dims. ds.dims={ds.dims}")

    # Ensure longitude convention matches
    # Uncomment if your file is 0–360 but CSV is −180–180:
    # if ds[lon_name].max() > 180 and df[lon_col].min() < 0:
    #     df[lon_col] = (df[lon_col] + 360) % 360

    lats = xr.DataArray(df[lat_col].values, dims="rows")
    lons = xr.DataArray(df[lon_col].values, dims="rows")

    if method == "nearest":
        vals = var_yr.sel({lat_name: lats, lon_name: lons}, method="nearest")
    else:
        vals = var_yr.interp({lat_name: lats, lon_name: lons}, method=method)

    df["bio_15"] = vals.values

    if out_csv:
        df.to_csv(out_csv, index=False)

    return df


In [9]:
df_out = add_bio10(
    csv_path="points_with_bio10.csv",
    nc_path="BIO15_era5_1979-2018_v1.0.nc",
    out_csv="points_with_bio10.csv"
)
